# PII Anonymization Leakage

Hashing an email address and calling the data "anonymized" is one of the most common
compliance mistakes in data engineering. When quasi-identifiers like ZIP code, date of
birth, and gender remain in cleartext, an attacker can cross-reference public datasets
and re-identify individuals — even without reversing the hash.

This notebook demonstrates the attack, measures k-anonymity, and shows how to
generalize quasi-identifiers until the data is genuinely safe.

In [ ]:
!pip install faker pandas dataprof --quiet

## Generate Synthetic User Data

We create 5 000 fake users with fields that look like a typical CRM export:
name, email, date of birth, ZIP code, gender, and a transaction amount.

ZIP codes are drawn from a small pool (~50) to simulate realistic low-cardinality
geographic data.

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from faker import Faker

fake = Faker("en_US")
Faker.seed(42)
np.random.seed(42)

N = 5_000

# Pool of ~50 ZIP codes to create realistic low cardinality
zip_pool = [fake.zipcode() for _ in range(50)]

users = pd.DataFrame({
    "name": [fake.name() for _ in range(N)],
    "email": [fake.unique.email() for _ in range(N)],
    "date_of_birth": [fake.date_of_birth(minimum_age=18, maximum_age=75) for _ in range(N)],
    "zip_code": np.random.choice(zip_pool, size=N),
    "gender": np.random.choice(["M", "F", "NB"], size=N, p=[0.48, 0.48, 0.04]),
    "transaction_amount": np.round(np.random.uniform(5, 2000, size=N), 2),
})

print(f"Users: {len(users)} rows")
display(users.head())

## Anti-Pattern: Hash the Email and Call It a Day

The naive approach: apply SHA-256 to the email (the only "direct identifier") and
leave everything else untouched. This feels secure — the hash is irreversible. But
the combination of quasi-identifiers (ZIP + DOB + gender) can uniquely identify
individuals.

Research by Latanya Sweeney showed that **87% of the US population** can be uniquely
identified by just {ZIP, DOB, gender}.

In [ ]:
# "Anonymize" by hashing emails
naive_anon = users.copy()
naive_anon["email"] = naive_anon["email"].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)
naive_anon = naive_anon.drop(columns=["name"])  # drop name too, feeling safe

print("Naively 'anonymized' data:")
display(naive_anon.head())

In [ ]:
# Measure k-anonymity on quasi-identifiers
quasi_ids = ["zip_code", "date_of_birth", "gender"]
group_sizes = naive_anon.groupby(quasi_ids).size()

k_min = group_sizes.min()
unique_count = (group_sizes == 1).sum()
total_groups = len(group_sizes)

print(f"k-anonymity (minimum group size): k = {k_min}")
print(f"Groups with k=1 (uniquely identifiable): {unique_count} / {total_groups} "
      f"({unique_count / total_groups * 100:.1f}%)")
print(f"\nIndividuals in k=1 groups: {unique_count} people can be re-identified")
print("\nk=1 means this person's quasi-identifier combo is unique in the dataset.")
print("An attacker with a voter registry can link the hash back to a real name.")

# Show the distribution
print("\nGroup size distribution:")
print(group_sizes.describe())

## Profiling with dataprof: Spotting Dangerous Cardinality

Even without manually checking k-anonymity, automated profiling can raise red flags.
Columns with high cardinality relative to the dataset size (like `date_of_birth`
with ~20,000 possible values over 5,000 rows) are inherently risky as
quasi-identifiers.

In [ ]:
import dataprof

profile = dataprof.profile(naive_anon.drop(columns=["email"]))  # profile the non-hashed columns
display(profile)

print("\nLook at 'date_of_birth' cardinality — almost as many distinct values as rows.")
print("Combined with 'zip_code' and 'gender', it creates a quasi-identifier fingerprint.")

## Correct Approach: k-Anonymity via Generalization

To achieve genuine anonymization, we must **reduce the precision** of quasi-identifiers
until every combination matches at least `k` individuals:

1. **date_of_birth** → generalize to **birth_year** (reduces ~20K values to ~57).
2. **zip_code** → generalize to **3-digit prefix** (reduces ~50 to ~40).
3. **Suppress** any remaining groups with fewer than `k` members.

A common threshold is **k ≥ 5**: any individual is indistinguishable from at least 4 others.

In [ ]:
K_THRESHOLD = 5

safe_anon = naive_anon.copy()

# Step 1: Generalize date_of_birth to birth_year
safe_anon["birth_year"] = pd.to_datetime(safe_anon["date_of_birth"]).dt.year
safe_anon = safe_anon.drop(columns=["date_of_birth"])

# Step 2: Generalize zip_code to 3-digit prefix
safe_anon["zip_3"] = safe_anon["zip_code"].astype(str).str[:3]
safe_anon = safe_anon.drop(columns=["zip_code"])

# Step 3: Suppress groups with k < threshold
generalized_qi = ["zip_3", "birth_year", "gender"]
group_sizes_new = safe_anon.groupby(generalized_qi).transform("size")
mask_safe = group_sizes_new >= K_THRESHOLD

n_suppressed = (~mask_safe).sum()
safe_anon = safe_anon[mask_safe].reset_index(drop=True)

print(f"Suppressed {n_suppressed} rows in groups with k < {K_THRESHOLD}")
print(f"Remaining rows: {len(safe_anon)}")

# Verify new k-anonymity
k_new = safe_anon.groupby(generalized_qi).size().min()
print(f"\nNew k-anonymity: k = {k_new} (target was k >= {K_THRESHOLD})")

print("\nSafe anonymized data:")
display(safe_anon.head(10))

In [ ]:
# Compare before/after
print("=== Before (naive hashing) ===")
qi_before = naive_anon.groupby(["zip_code", "date_of_birth", "gender"]).size()
print(f"  k-anonymity:       k = {qi_before.min()}")
print(f"  Unique individuals: {(qi_before == 1).sum()}")
print(f"  Total rows:        {len(naive_anon)}")

print(f"\n=== After (generalization + suppression, k >= {K_THRESHOLD}) ===")
qi_after = safe_anon.groupby(generalized_qi).size()
print(f"  k-anonymity:       k = {qi_after.min()}")
print(f"  Unique individuals: {(qi_after == 1).sum()}")
print(f"  Total rows:        {len(safe_anon)}")
print(f"  Data loss:         {n_suppressed} rows ({n_suppressed / len(naive_anon) * 100:.1f}%)")

## Takeaways

| Technique | Protects Against | Risk If Skipped |
| :--- | :--- | :--- |
| Hash direct identifiers (email, name) | Direct lookup attacks | Identity exposed in cleartext |
| Generalize quasi-identifiers (DOB → year, ZIP → prefix) | Linkage attacks via cross-referencing | 87% of US population uniquely identifiable by {ZIP, DOB, gender} |
| Suppress small groups (k < threshold) | Residual re-identification in rare combos | Outlier individuals remain unique |
| Profile with dataprof / audit cardinality | Overlooked high-cardinality columns | Hidden quasi-identifiers slip through |

### Legal context

Under GDPR, **pseudonymization** (e.g., hashing emails) still counts as personal data
processing and requires a legal basis. Only **anonymization** — where re-identification is
"reasonably impossible" — falls outside GDPR scope. k-anonymity with sufficient generalization
is one recognized approach, alongside l-diversity and t-closeness for sensitive attributes.

**Bottom line:** hashing alone is pseudonymization, not anonymization. If quasi-identifiers
remain, you still have personal data in the eyes of the law.